# Cleaning Canadian Immigration Data (UN 2015 Revision)

**Goal**  
The original Excel file from the United Nations contains many header rows and mixed metadata.  
This notebook reads the file, removes the unnecessary rows, sets clear column names, and produces a clean DataFrame that is ready for analysis.

**Data source**  
United Nations, Department of Economic and Social Affairs, Population Division (2015).  
International Migration Flows to and from Selected Countries: The 2015 Revision.  
(United Nations database, POP/DB/MIG/Flow/Rev.2015)

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np

# Make sure all columns are visible when we print
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

## 2. Load the raw Excel file

The useful data is on the sheet named **"Canada by Citizenship"**.  
We first read the whole sheet without assuming any header so we can see the structure.

In [ ]:
# Read the second sheet, no header yet
raw = pd.read_excel(
    'Canada.xlsx',
    sheet_name='Canada by Citizenship',
    header=None
)

print('Raw shape:', raw.shape)
print('\nFirst 25 rows (first 10 columns):')
raw.iloc[:25, :10]

## 3. Identify the real header row

Looking at the output above, the actual column names sit on **row 20** (0-based index).  
Everything above that row is title / metadata and can be skipped.

In [ ]:
# Row 20 contains the proper column names
print('Header row content:')
print(list(raw.iloc[20]))

## 4. Create the clean DataFrame

Steps:
1. Skip the first 20 rows (metadata).
2. Use row 20 as the header.
3. Rename the year columns from float (1980.0) to integer (1980).
4. Replace the ".." missing-value markers with NaN.
5. Convert the year columns to numeric.

In [ ]:
# Read again, this time skipping the metadata rows and using the correct header
df = pd.read_excel(
    'Canada.xlsx',
    sheet_name='Canada by Citizenship',
    skiprows=20,          # skip the first 20 rows of titles/notes
    header=0              # the next row becomes the header
)

# Show the column names we obtained
print('Columns after reading:')
print(df.columns.tolist())
print('\nShape:', df.shape)
df.head()

## 5. Clean column names and data types

In [ ]:
# The year columns currently look like 1980.0, 1981.0, ...
# Convert them to plain integers so they are easier to work with later
year_cols = [col for col in df.columns if isinstance(col, (int, float))]
rename_dict = {col: int(col) for col in year_cols}
df = df.rename(columns=rename_dict)

print('Year columns after renaming:')
print([c for c in df.columns if isinstance(c, int)])

In [ ]:
# Replace the ".." markers that mean "no data" with proper NaN
df = df.replace('..', np.nan)

# Convert all year columns to numeric (float) so we can do calculations
years = list(range(1980, 2014))
for y in years:
    if y in df.columns:
        df[y] = pd.to_numeric(df[y], errors='coerce')

print('Data types of the first few year columns:')
print(df[years[:5]].dtypes)

## 6. Final clean DataFrame

At this point the DataFrame has a clear structure:

| Column       | Meaning                          |
|--------------|----------------------------------|
| Type         | Immigrants                       |
| Coverage     | Foreigners / Citizens / Both     |
| OdName       | Country or region name           |
| AREA         | Numeric area code                |
| AreaName     | Continent / major area           |
| REG          | Numeric region code              |
| RegName      | More detailed region             |
| DEV          | Development region code          |
| DevName      | Developed / Developing           |
| 1980 … 2013  | Number of immigrants that year   |

In [ ]:
print('Final shape:', df.shape)
print('\nColumn list:')
print(df.columns.tolist())
print('\nFirst 10 rows:')
df.head(10)

In [ ]:
# Quick check – total immigrants for a few well-known countries
print('Sample totals (sum of all years):')
sample = df[df['OdName'].isin(['India', 'China', 'United Kingdom of Great Britain and Northern Ireland', 'Philippines'])].copy()
sample['Total'] = sample[years].sum(axis=1)
sample[['OdName', 'Total']].sort_values('Total', ascending=False)

## 7. Save the cleaned data (optional)

You can save the result as a CSV for later use.

In [ ]:
# Uncomment the next line if you want to write a clean CSV
# df.to_csv('Canada_cleaned.csv', index=False)

print('Cleaning finished. The DataFrame "df" is ready for analysis.')